# Tools

Models can request to all tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
- A Schema, including the name of the tool, a description, and/or argument definitions ( often a JSON schema )
- A function or co-routing to execute


In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

groq = init_chat_model('groq:qwen/qwen3-32b')
groq

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CE712CFE00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CE71488C20>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def divide(a: int, b: int) -> int:
    """Divide two numbers."""
    return a / b

@tool
def get_weather(location:str)->str:
    """Get the weather for a location."""
    return f"The weather in {location} is sunny."

tools = [add, subtract, multiply, divide,get_weather]

groq_with_tools = groq.bind_tools(tools)


In [3]:
response = groq_with_tools.invoke('what is weather in delhi')
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Delhi. Let me check the available tools. There\'s a function called get_weather that takes a location parameter. Since Delhi is the location specified, I should call that function with "Delhi" as the argument. I don\'t need to use any other functions because the query is straightforward and only requires the weather information. Let me make sure the parameters are correctly formatted as per the function\'s requirements.\n', 'tool_calls': [{'id': '5sem5888k', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 410, 'total_tokens': 525, 'completion_time': 0.174177359, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.017444913, 'prompt_tokens_details': None, 'queue_time': 0.046002606, 'total_time': 0.191622272}, 'model_name': 'qwen/qwen3-32b', 'system_

## Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{'role': 'user', 'content': 'what is weather in delhi ?'}]
ai_msg = groq_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
# Step 3: Pass results back to model for final response
final_response = groq_with_tools.invoke(messages)
print(final_response.content)

The weather in Delhi is currently sunny. Let me know if you need more details!
